<a href="https://colab.research.google.com/github/ChaimElchik/GPS-Demo/blob/main/GPS_DEM_DepthAnythingDistanceSamplingVideoSequenceFullV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Elephant Detection and GPS Localization Video Sequence Full 🐘📍


---
## Setup Environment

In [ ]:
# Install necessary packages
!pip install piexif geopy pyproj torch torchvision transformers timm accelerate -q
!pip install ultralytics==8.3.18 --upgrade --quiet
!apt-get install -y exiftool -qq

# Import necessary libraries
import cv2
import torch
import numpy as np
from ultralytics import YOLO
import pandas as pd
import time
import os
from pathlib import Path
import matplotlib.pyplot as plt
from pyproj import Transformer
import math
import re
from datetime import datetime, timedelta
import subprocess
import json
import csv
from google.colab import files
from PIL import Image
import io
from geopy.distance import geodesic
from IPython.display import Image as IPImage, display

# Create output directories
os.makedirs("Detections", exist_ok=True)
os.makedirs("Processed_Output", exist_ok=True)

print("\nSetup Complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.6/876.6 kB 14.4 MB/s eta 0:00:00
Selecting previously unselected package libarchive-zip-perl.
(Reading database ... 126281 files and directories currently 


##  1. Upload Model, SRT File, Video File and Tracker Config File

In [ ]:
from google.colab import files
import os

print("--- Step 1: Upload Files ---")
print("Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').")

# Upload files
uploaded = files.upload()

# Store paths to the uploaded files
video_path = None
srt_path = None
model_path = None
tracker_config_path = None

for fn in uploaded.keys():
    if fn.lower().endswith('.pt'):
        model_path = fn
        print(f"✅ Model file '{fn}' found.")
    elif fn.lower().endswith('.yaml'):
        tracker_config_path = fn
        print(f"✅ Tracker config file '{fn}' found.")
    elif fn.lower().endswith(('.mp4', '.mov', '.avi')):
        video_path = fn
        print(f"✅ Video file '{fn}' found.")
    elif fn.lower().endswith('.srt'):
        srt_path = fn
        print(f"✅ SRT file '{fn}' found.")

# --- Verification ---
print("\n--- Verifying files ---")
if not video_path: print("❌ ERROR: Video file not uploaded.")
if not srt_path: print("❌ ERROR: SRT file not uploaded.")
if not model_path: print(f"❌ ERROR: Model .pt file not uploaded.")
if not tracker_config_path: print("❌ ERROR: Tracker config .yaml file not uploaded.")

if video_path and srt_path and model_path and tracker_config_path:
    # This global variable is used by the next cell.
    MODEL_PATH = model_path
    print("\n--- All files ready for processing! ---")

--- Step 1: Upload Files ---
Please upload your video, SRT file, your .pt model, and your tracker config (e.g., 'botsort.yaml').


Saving best.pt to best.pt
Saving botsortV5.yaml to botsortV5.yaml
Saving DJI_20250618120033_0001_D.MP4 to DJI_20250618120033_0001_D.MP4
Saving DJI_20250618120033_0001_D.SRT to DJI_20250618120033_0001_D.SRT
✅ Model file 'best.pt' found.
✅ Tracker config file 'botsortV5.yaml' found.
✅ Video file 'DJI_20250618120033_0001_D.MP4' found.
✅ SRT file 'DJI_20250618120033_0001_D.SRT' found.

--- Verifying files ---

--- All files ready for processing! ---


---
## 2. Core Logic and Helper Functions


In [ ]:
from PIL import Image

# --- Dependencies Check ---
try:
    from ultralytics import YOLO
    from geopy.distance import geodesic
    from geopy.point import Point
except ImportError as e:
    print(f"ERROR: Missing dependency - {e}. Please install required libraries.")
    print("Run: pip install ultralytics opencv-python pyproj geopy requests torch torchvision transformers timm accelerate Pillow")
    exit()

# --- Global Configuration ---
OUTPUT_DIR = "Video_Processing_Output_Full"
# MODEL_PATH is set dynamically in the previous cell.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEPTH_MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'

# --- SENSOR CONFIGURATION ---
SENSOR_WIDTH_MM = 17.3
SENSOR_HEIGHT_MM = 13.0
print(f"INFO: Using Sensor Size {SENSOR_WIDTH_MM}mm x {SENSOR_HEIGHT_MM} (Mavic 3 Pro Main Cam).")
print(f"INFO: Using device: {DEVICE} for deep learning models.")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- SRT Parsing & Geolocation Functions ---
def parse_srt_file(srt_path):
    print(f"INFO: Parsing SRT file: {srt_path}")
    metadata_map = {}
    with open(srt_path, 'r') as f: content = f.read()
    pattern = re.compile(
        r"FrameCnt: (\d+).*?\[focal_len: ([\d\.]+)\]"
        r".*?\[latitude: ([\d\.\-]+)\] \[longitude: ([\d\.\-]+)\] "
        r"\[rel_alt: ([\d\.\-]+) abs_alt: ([\d\.\-]+)\] "
        r"\[gb_yaw: ([\d\.\-]+) gb_pitch: ([\d\.\-]+) gb_roll: ([\d\.\-]+)\]", re.DOTALL)
    for match in pattern.finditer(content):
        frame_cnt = int(match.group(1))
        metadata_map[frame_cnt] = {
            'focal_len': float(match.group(2)), 'latitude': float(match.group(3)),
            'longitude': float(match.group(4)), 'rel_alt': float(match.group(5)),
            'abs_alt': float(match.group(6)), 'gb_yaw': float(match.group(7)),
            'gb_pitch': float(match.group(8)), 'gb_roll': float(match.group(9)),}
    print(f"✅ Successfully parsed metadata for {len(metadata_map)} frames from SRT.")
    return metadata_map

def get_depth_map(frame, model, processor):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    prediction = torch.nn.functional.interpolate(
        outputs.predicted_depth.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False)
    return prediction.squeeze().cpu().numpy()

def get_dem_elevation_from_api(latitude, longitude):
    try:
        url = f"https://api.opentopodata.org/v1/eudem25m?locations={latitude},{longitude}"
        response = requests.get(url, verify=False, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data['results'] and data['results'][0]['elevation'] is not None:
                return data['results'][0]['elevation']
    except requests.exceptions.RequestException: return None
    return None

def get_camera_intrinsics(f_mm, s_w_mm, s_h_mm, i_w, i_h):
    fx = i_w * f_mm / s_w_mm; fy = i_h * f_mm / s_h_mm
    cx, cy = i_w / 2, i_h / 2
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

def get_rotation_matrix(pitch_deg, yaw_deg, roll_deg):
    yaw, pitch, roll = map(math.radians, [yaw_deg, pitch_deg, roll_deg])
    Rz = np.array([[math.cos(yaw), -math.sin(yaw), 0], [math.sin(yaw), math.cos(yaw), 0], [0, 0, 1]])
    Ry = np.array([[math.cos(pitch), 0, math.sin(pitch)], [0, 1, 0], [-math.sin(pitch), 0, math.cos(pitch)]])
    Rx = np.array([[1, 0, 0], [0, math.cos(roll), -math.sin(roll)], [0, math.sin(roll), math.cos(roll)]])
    R_gimbal = Rz @ Ry @ Rx
    R_cam_to_body = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]]).T
    return R_gimbal @ R_cam_to_body

def calculate_destination_gps(origin_lat, origin_lon, east_m, north_m):
    bearing = math.degrees(math.atan2(east_m, north_m))
    distance_meters = math.hypot(east_m, north_m)
    destination = geodesic(meters=distance_meters).destination(Point(origin_lat, origin_lon), bearing)
    return destination.latitude, destination.longitude

def image_point_to_gps_from_depth(u, v, K, R, o_lat, o_lon, abs_depth_map):
    v_idx, u_idx = int(round(v)), int(round(u))
    if not (0 <= v_idx < abs_depth_map.shape[0] and 0 <= u_idx < abs_depth_map.shape[1]): return None, None
    distance_to_target = abs_depth_map[v_idx, u_idx]
    K_inv = np.linalg.inv(K)
    ray_cam = K_inv @ np.array([u, v, 1])
    ray_cam_unit = ray_cam / np.linalg.norm(ray_cam)
    point_in_cam_coords = ray_cam_unit * distance_to_target
    ned_offsets = R @ point_in_cam_coords
    ned_n, ned_e = ned_offsets[0], ned_offsets[1]
    return calculate_destination_gps(o_lat, o_lon, ned_e, ned_n)

# --- Drawing & Saving Functions ---
def draw_overlays(frame, tracked_objects_data):
    frame_height, frame_width, _ = frame.shape
    cv2.line(frame, (frame_width // 2, 0), (frame_width // 2, frame_height), (255, 0, 0), 2)
    for data in tracked_objects_data:
        x1, y1, x2, y2 = data['box']
        obj_id = data['id']; conf = data['conf']
        cv2.rectangle(frame, (x1, y1), (x2, y2), (128, 0, 128), 2)
        text = f"ID: {obj_id} | Conf: {conf:.2f}"
        if 'gps' in data:
            lat, lon = data['gps']
            text += f" | GPS: {lat:.5f}, {lon:.5f}"
        if 'transect_dist' in data:
            center_u, center_v = int((x1 + x2) / 2), int((y1 + y2) / 2)
            cv2.line(frame, (center_u, center_v), (frame_width // 2, center_v), (0, 0, 255), 1)
            text += f" | Dist: {data['transect_dist']:.2f}m"
        cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (128, 0, 128), 2)
    return frame

def save_frame_by_frame_log(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'frame_by_frame_log.csv')
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['Frame', 'Timestamp', 'ObjectID', 'Latitude', 'Longitude', 'Confidence', 'center_u', 'center_v'])
        for result in all_results:
            writer.writerow([result['frame'], result['timestamp'], result['id'],
                f"{result['lat']:.6f}", f"{result['lon']:.6f}", f"{result['conf']:.4f}",
                f"{result['center_u']:.2f}", f"{result['center_v']:.2f}"])
    print(f"✅ Frame-by-frame log saved to: {filepath}")

def save_unique_objects_summary(all_results):
    filepath = os.path.join(OUTPUT_DIR, 'unique_objects_first_seen.csv')
    first_seen = {}
    for result in all_results:
        obj_id = result['id']
        if obj_id not in first_seen:
            first_seen[obj_id] = {'id': obj_id, 'timestamp': result['timestamp'],
                'lat': result['lat'], 'lon': result['lon'], 'conf': result['conf']}
    with open(filepath, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['ObjectID', 'TimestampFirstSeen', 'Latitude', 'Longitude', 'Confidence'])
        for obj_id in sorted(first_seen.keys()):
            data = first_seen[obj_id]
            writer.writerow([data['id'], data['timestamp'], f"{data['lat']:.6f}",
                f"{data['lon']:.6f}", f"{data['conf']:.4f}"])
    print(f"✅ Unique objects summary saved to: {filepath}")

INFO: Using Sensor Size 17.3mm x 13.0 (Mavic 3 Pro Main Cam).
INFO: Using device: cpu for deep learning models.


---
## 3. Main Execution Block

In [ ]:
def main_full_video_pipeline(video_path, srt_path, model_path, tracker_config):
    start_time = time.time()
    print("\n--- 🚀 Starting FULL Video Processing Pipeline 🚀 ---")

    try:
        yolo_model = YOLO(model_path)
        depth_processor = AutoImageProcessor.from_pretrained(DEPTH_MODEL_NAME)
        depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL_NAME).to(DEVICE)
        srt_metadata = parse_srt_file(srt_path)
    except Exception as e:
        print(f"❌ FATAL ERROR during initialization: {e}")
        return

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"❌ ERROR: Cannot open video file {video_path}")
        return

    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"INFO: Video Properties: {frame_width}x{frame_height} @ {fps:.2f} FPS, {total_frames} total frames.")

    output_video_path = os.path.join(OUTPUT_DIR, 'annotated_video_full.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_video = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))
    print(f"INFO: Output video will be saved to: {output_video_path}")

    frame_count = 0
    all_frame_results = []

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1
        timestamp = frame_count / fps
        print(f"\n--- Processing Frame {frame_count}/{total_frames} (Timestamp: {timestamp:.2f}s) ---")

        if frame_count not in srt_metadata:
            print(f"⚠️ WARNING: No metadata in SRT for frame {frame_count}. Writing original frame.")
            out_video.write(frame)
            continue
        meta = srt_metadata[frame_count]

        results = yolo_model.track(frame, persist=True, tracker=tracker_config, conf=0.5, verbose=False)

        tracked_objects_data = []
        for result in results:
            if result.boxes.id is not None:
                boxes = result.boxes.xyxy.cpu().numpy().astype(int)
                ids = result.boxes.id.cpu().numpy().astype(int)
                confs = result.boxes.conf.cpu().numpy()
                for i in range(len(ids)):
                    tracked_objects_data.append({'id': ids[i], 'box': boxes[i], 'conf': confs[i]})

        if not tracked_objects_data:
            print("INFO: No objects detected in this frame.")
            cv2.line(frame, (frame_width // 2, 0), (frame_width // 2, frame_height), (255, 0, 0), 2)
            out_video.write(frame)
            continue

        print(f"INFO: Tracking {len(tracked_objects_data)} objects.")

        try:
            # --- FULL PROCESSING: Always recalculate depth and geolocation data ---
            ground_elevation = get_dem_elevation_from_api(meta['latitude'], meta['longitude'])
            base_agl = (meta['abs_alt'] - ground_elevation) if ground_elevation is not None else meta['rel_alt']
            relative_depth_map = get_depth_map(frame, depth_model, depth_processor)
            scale_factor = base_agl / np.mean(relative_depth_map)
            absolute_depth_map = relative_depth_map * scale_factor

            K = get_camera_intrinsics(meta['focal_len'], SENSOR_WIDTH_MM, SENSOR_HEIGHT_MM, frame_width, frame_height)
            R = get_rotation_matrix(meta['gb_pitch'], meta['gb_yaw'], meta['gb_roll'])
            gsd = (base_agl * SENSOR_WIDTH_MM) / (meta['focal_len'] * frame_width)

            for obj_data in tracked_objects_data:
                box = obj_data['box']
                center_u, center_v = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2

                pixel_distance = abs(center_u - (frame_width / 2))
                obj_data['transect_dist'] = pixel_distance * gsd

                lat, lon = image_point_to_gps_from_depth(center_u, center_v, K, R, meta['latitude'], meta['longitude'], absolute_depth_map)
                if lat is not None and lon is not None:
                    obj_data['gps'] = (lat, lon)
                    all_frame_results.append({
                        'frame': frame_count, 'timestamp': f"{timestamp:.3f}", 'id': obj_data['id'],
                        'lat': lat, 'lon': lon, 'conf': obj_data['conf'],
                        'center_u': center_u, 'center_v': center_v
                    })

            annotated_frame = draw_overlays(frame.copy(), tracked_objects_data)
            out_video.write(annotated_frame)

        except Exception as e:
            print(f"❌ ERROR processing frame {frame_count}: {e}")
            traceback.print_exc()
            out_video.write(frame)
            continue

    cap.release()
    out_video.release()
    print("\n\n--- ✅ Full Video Processing Complete ---")

    if all_frame_results:
        save_frame_by_frame_log(all_frame_results)
        save_unique_objects_summary(all_frame_results)
        print(f"✅ Annotated video saved to: {output_video_path}")
    else:
        print("INFO: No objects were successfully geolocated in the video.")

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"\nTotal process completed in {elapsed_time:.2f} seconds.")
    if total_frames > 0:
        print(f"Average time per frame: {elapsed_time / total_frames:.3f} seconds.")

if 'video_path' in locals() and video_path and 'srt_path' in locals() and srt_path and 'model_path' in locals() and model_path and 'tracker_config_path' in locals() and tracker_config_path:
    main_full_video_pipeline(video_path, srt_path, model_path, tracker_config_path)
else:
    print("\n❌ Please run Cell 1 to upload all required files before running this cell.")



--- 🚀 Starting Video Processing Pipeline 🚀 ---


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import os

csv_path = os.path.join('Video_Processing_Output_Full', 'unique_objects_first_seen.csv')
print(f"--- Loading results from: {csv_path} ---")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("✅ Successfully loaded the summary of unique object detections:")
    display(df)
else:
    print(f"❌ ERROR: The output file was not found at '{csv_path}'.")
    print("Please ensure the main processing cell (CELL 3) completed without errors.")


In [ ]:
import pandas as pd
import os
from geopy.distance import geodesic

log_path = os.path.join('Video_Processing_Output_Full', 'frame_by_frame_log.csv')
print(f"--- Loading detailed log from: {log_path} ---")

if os.path.exists(log_path):
    try:
        df_log = pd.read_csv(log_path)
        print("✅ Successfully loaded the frame-by-frame log.")
        print("\n--- Calculating object movement... ---")

        df_log = df_log.sort_values(by=['ObjectID', 'Timestamp'])

        movement_records = []
        for object_id, group in df_log.groupby('ObjectID'):
            group['Prev_Latitude'] = group['Latitude'].shift(1)
            group['Prev_Longitude'] = group['Longitude'].shift(1)
            for index, row in group.iterrows():
                if pd.notna(row['Prev_Latitude']):
                    distance_moved = geodesic((row['Prev_Latitude'], row['Prev_Longitude']), (row['Latitude'], row['Longitude'])).meters
                    if distance_moved > 0.01:
                        movement_records.append({'ObjectID': row['ObjectID'], 'DistanceMovedMeters': distance_moved})

        if movement_records:
            df_movement = pd.DataFrame(movement_records)

            total_distance_records = []
            for object_id, group in df_log.groupby('ObjectID'):
                if len(group) > 1:
                    first_row = group.iloc[0]
                    start_pos = (first_row['Latitude'], first_row['Longitude'])
                    max_distance = 0
                    frame_at_max_distance = first_row['Frame']
                    for index, row in group.iloc[1:].iterrows():
                        distance = geodesic(start_pos, (row['Latitude'], row['Longitude'])).meters
                        if distance > max_distance:
                            max_distance = distance
                            frame_at_max_distance = row['Frame']
                    total_distance_records.append({'ObjectID': object_id, 'TotalDistanceMeters': max_distance, 'FrameAtMaxDistance': frame_at_max_distance})

            df_total_dist = pd.DataFrame(total_distance_records)
            df_avg_move = df_movement.groupby('ObjectID')['DistanceMovedMeters'].mean().reset_index()
            df_avg_move.rename(columns={'DistanceMovedMeters': 'AverageMoveMeters'}, inplace=True)

            if not df_total_dist.empty:
                df_summary = pd.merge(df_total_dist, df_avg_move, on='ObjectID', how='left')
                print("\n--- Summary of Movement per Object ---")
                display(df_summary.set_index('ObjectID'))
            else:
                 print("\n--- Summary of Movement per Object (No total distance calculated) ---")
                 display(df_avg_move.set_index('ObjectID'))
        else:
            print("\n-> No significant object movement was detected between frames.")
    except Exception as e:
        print(f"❌ An error occurred during movement calculation: {e}")
        traceback.print_exc()
else:
    print(f"❌ ERROR: The detailed log file was not found at '{log_path}'.")


In [ ]:
import pandas as pd
import os
import requests
import cv2
from pathlib import Path

print("--- Starting Distance Sampling Analysis ---")

log_path = os.path.join('Video_Processing_Output_Full', 'frame_by_frame_log.csv')
video_path_for_props = video_path
srt_path_for_meta = srt_path
output_csv_path = os.path.join('Video_Processing_Output_Full', 'distance_sampling_data.csv')

def get_region_from_gps_once(lat, lon):
    print(f"\nINFO: Querying OpenStreetMap API for region name at {lat:.4f}, {lon:.4f}...")
    headers = {'User-Agent': 'EcologicalSurveyScript/1.0'}
    url = f"https://nominatim.openstreetmap.org/reverse?format=json&lat={lat}&lon={lon}&zoom=10"
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            address = data.get('address', {})
            name_parts = [address.get('nature_reserve'), address.get('national_park'), address.get('village'),
                          address.get('town'), address.get('city'), address.get('state'), address.get('country')]
            region_name = ', '.join(part for part in name_parts if part)
            if region_name:
                print(f"INFO: API lookup successful. Region: {region_name}")
                return region_name
    except requests.exceptions.RequestException as e:
        print(f"WARNING: Region API request failed. Error: {e}")
    return None

if os.path.exists(log_path):
    try:
        df_log = pd.read_csv(log_path)
        required_columns = ['Latitude', 'Longitude', 'Frame', 'ObjectID', 'center_u']
        if not all(col in df_log.columns for col in required_columns):
            print(f"❌ ERROR: Log file missing required columns. Found: {df_log.columns.to_list()}")
        else:
            print("✅ Log file loaded successfully.")
            srt_data = parse_srt_file(srt_path_for_meta)
            cap = cv2.VideoCapture(video_path_for_props)
            frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            cap.release()

            first_lat, first_lon = df_log.iloc[0]['Latitude'], df_log.iloc[0]['Longitude']
            region_label = get_region_from_gps_once(first_lat, first_lon)
            if not region_label:
                region_label = Path(video_path_for_props).stem

            distance_sampling_records = []
            for index, row in df_log.iterrows():
                frame_num = row['Frame']
                if frame_num in srt_data:
                    meta = srt_data[frame_num]
                    gsd_w = (meta['rel_alt'] * SENSOR_WIDTH_MM) / (meta['focal_len'] * frame_width)
                    gsd_h = (meta['rel_alt'] * SENSOR_HEIGHT_MM) / (meta['focal_len'] * frame_height)
                    meter_distance = abs(row['center_u'] - (frame_width / 2)) * gsd_w
                    area_sq_km = ((frame_width * gsd_w) * (frame_height * gsd_h)) / 1_000_000

                    distance_sampling_records.append({
                        'Region.Label': region_label, 'Area.km2': f"{area_sq_km:.6f}",
                        'Sample.Label': f"frame_{frame_num}", 'Effort.m': f"{frame_height * gsd_h:.2f}",
                        'object': row['ObjectID'], 'distance': f"{meter_distance:.2f}", 'size': 1,
                        'Lat': f"{row['Latitude']:.6f}", 'Lon': f"{row['Longitude']:.6f}"})

            if distance_sampling_records:
                df_dist_sample = pd.DataFrame(distance_sampling_records)
                df_dist_sample.to_csv(output_csv_path, index=False)
                print(f"\n✅ Distance sampling data saved to: {output_csv_path}")
                print("\n--- First 5 rows of the distance sampling data ---")
                display(df_dist_sample.head())
            else:
                print("\n-> No data to process for distance sampling.")
    except Exception as e:
        print(f"❌ An error occurred during distance sampling analysis: {e}")
        traceback.print_exc()
else:
    print(f"❌ ERROR: The detailed log file was not found at '{log_path}'.")
